# Tabular Deep Learning

The dominant practitioner wisdom for tabular data is: start with gradient-boosted trees. XGBoost and LightGBM are fast, require minimal preprocessing, handle mixed feature types natively, and routinely outperform neural networks on medium-sized datasets with irregular decision boundaries. Yet neural methods have carved out a real niche — datasets with hundreds of thousands of rows, smooth target functions, rich categorical features with transferable structure, or settings where pretraining is available. This notebook examines the bridge: what it takes to build a competitive MLP baseline, how to handle categorical features with learned embeddings, and how architectures like TabNet and FT-Transformer attempt to close the gap with trees. We conclude with an evidence-based discussion of when to reach for neural methods and when not to.

## MLP Baselines for Tabular Data

Why does a naive MLP lose to XGBoost on most tabular benchmarks? The core issue is **inductive bias**. Decision trees partition the feature space with axis-aligned cuts — exactly the geometry that arises naturally from heterogeneous tabular features where each column has independent semantics. An MLP has no such bias: it applies dense linear projections that mix all features at every layer, which is the wrong structure for data where most features are irrelevant to any given prediction, interactions are sparse and feature-specific, and the target function has irregular discontinuities aligned with individual features.

There are also practical obstacles. MLPs are sensitive to feature scale: a raw housing dataset with `median_income` in the range $[0, 15]$ and `population` in $[3, 35000]$ will cause gradient magnitudes to vary by orders of magnitude, destabilizing training. Trees are invariant to monotone rescaling of any feature. Similarly, MLPs have no native mechanism for categorical features — a tree can split on `occupation == 'Prof-specialty'` directly, while an MLP requires some encoding that preserves meaningful distance structure.

**The right MLP baseline.** A naive `Linear → ReLU` stack is not a fair comparison. A properly configured MLP for tabular data includes (1) **StandardScaler** on all continuous features before the network sees them; (2) **`nn.BatchNorm1d`** after each linear layer to normalize activations across the batch, reducing sensitivity to initialization and scale; (3) **weight decay** via AdamW to regularize the dense weight matrices; (4) a **cosine annealing** learning rate schedule to avoid premature convergence; and (5) **early stopping** on a held-out validation set to prevent overfitting. Even with all of these, an MLP without architecture search rarely beats a well-tuned XGBoost on medium-sized datasets. The value of the MLP baseline is to establish a fair lower bound for neural methods before reaching for more complex architectures.

Setting up imports and loading the California housing dataset:

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats("svg")

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data = fetch_california_housing()
X, y = data.data.astype(np.float32), data.target.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=SEED
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f"train: {X_train.shape}  val: {X_val.shape}  test: {X_test.shape}")

**Model.** The MLP uses three hidden layers of width 256 with BatchNorm and Dropout after each ReLU:

In [ ]:
def make_mlp(d_in, hidden=256, n_layers=3, dropout=0.1):
    layers = []
    in_dim = d_in
    for _ in range(n_layers):
        layers += [
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
        ]
        in_dim = hidden
    layers.append(nn.Linear(hidden, 1))
    return nn.Sequential(*layers)

mlp = make_mlp(d_in=X_train.shape[1]).to(DEVICE)
print(mlp)

**Training.** We use AdamW with cosine annealing and early stopping on validation RMSE:

In [ ]:
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32, device=DEVICE)

Xtr = to_tensor(X_train); ytr = to_tensor(y_train).unsqueeze(1)
Xvl = to_tensor(X_val);   yvl = to_tensor(y_val).unsqueeze(1)
Xte = to_tensor(X_test)

optimizer = torch.optim.AdamW(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=200, eta_min=1e-5
)
criterion = nn.MSELoss()

BATCH  = 512
EPOCHS = 200
PATIENCE = 20

best_val_rmse = float("inf")
patience_counter = 0
best_state = None
train_losses, val_rmses = [], []

n = Xtr.shape[0]
for epoch in range(EPOCHS):
    mlp.train()
    perm = torch.randperm(n, device=DEVICE)
    epoch_loss = 0.0
    for i in range(0, n, BATCH):
        idx = perm[i:i + BATCH]
        xb, yb = Xtr[idx], ytr[idx]
        optimizer.zero_grad()
        loss = criterion(mlp(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    scheduler.step()
    train_losses.append(epoch_loss / n)

    mlp.eval()
    with torch.no_grad():
        val_pred = mlp(Xvl).squeeze().cpu().numpy()
    val_rmse = root_mean_squared_error(y_val, val_pred)
    val_rmses.append(val_rmse)

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        patience_counter = 0
        best_state = {k: v.clone() for k, v in mlp.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stop at epoch {epoch + 1}")
            break

mlp.load_state_dict(best_state)
mlp.eval()
with torch.no_grad():
    test_pred_mlp = mlp(Xte).squeeze().cpu().numpy()

mlp_rmse = root_mean_squared_error(y_test, test_pred_mlp)
print(f"MLP test RMSE: {mlp_rmse:.4f}")

The learning curves illustrate the training dynamics:

In [ ]:
#| code-fold: true
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(train_losses, color="C0", linewidth=1.5)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("MSE loss (train)")
axes[0].set_title("Training loss")
axes[0].grid(linestyle="dotted", alpha=0.6)

axes[1].plot(val_rmses, color="C1", linewidth=1.5)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("RMSE (val)")
axes[1].set_title("Validation RMSE")
axes[1].grid(linestyle="dotted", alpha=0.6)
plt.tight_layout();

**Figure.** Training loss decreases monotonically while validation RMSE plateaus and then flattens — early stopping prevents overfitting. The cosine schedule produces a smooth decay without the sharp drops seen with step-based schedules.

For comparison we train XGBoost on the same split with default hyperparameters and minimal tuning:

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    early_stopping_rounds=20,
    eval_metric="rmse",
    verbosity=0,
)
xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

xgb_rmse = root_mean_squared_error(y_test, xgb.predict(X_test))
print(f"XGBoost test RMSE: {xgb_rmse:.4f}")
print(f"MLP     test RMSE: {mlp_rmse:.4f}")
print(f"MLP relative gap:  {(mlp_rmse - xgb_rmse) / xgb_rmse * 100:+.1f}%")

**Result.** XGBoost typically wins by 5–15% RMSE on California housing at this dataset size. The gap illustrates why the default practitioner workflow starts with trees: even a properly configured MLP requires significant architecture search to match them.

:::{.callout-caution}
BatchNorm interacts poorly with very small batch sizes (fewer than ~16 samples): the batch statistics become noisy estimates of the population statistics, and training becomes unstable. If memory constraints force small batches, consider LayerNorm or GroupNorm instead — both are independent of batch size and compatible with tabular data.

:::

## Embedding Categorical Features

The standard approach to categorical features in classical ML is one-hot encoding: a column `occupation` with cardinality $C$ becomes $C$ binary columns, one per category. One-hot works well for low-cardinality features but breaks down when $C$ is large — a feature with $C = 1000$ categories adds 1000 columns, most containing zeros for any given sample. These sparse, high-dimensional representations are hard to regularize and do not generalize well across categories with few training examples.

**Learned embeddings.** An `nn.Embedding` layer is a learnable lookup table $E \in \mathbb{R}^{C \times d_e}$ that maps each category index $c \in \{0, \ldots, C-1\}$ to a dense vector $\mathbf{e}_c \in \mathbb{R}^{d_e}.$ The embedding dimension $d_e$ is a hyperparameter: the common heuristic is

$$d_e = \min\!\left(600, \left\lfloor \frac{C}{2} \right\rfloor\right),$$

which allocates more capacity to high-cardinality features while avoiding excessive width. The embedding is trained end-to-end alongside the rest of the network, so the geometry of embedding space is shaped by the prediction task: categories that behave similarly for the target variable end up close together.

The full model concatenates the embeddings of all categorical features with the (normalized) continuous features before feeding the combined vector into the MLP:

$$\mathbf{h}_0 = \left[\mathbf{e}_{c_1}^\top \;\bigg|\; \cdots \;\bigg|\; \mathbf{e}_{c_K}^\top \;\bigg|\; \mathbf{x}_{\text{cont}}^\top\right] \in \mathbb{R}^{\sum_k d_{e_k} + d_{\text{cont}}}.$$

This is the architecture used by the Rossman store sales Kaggle winners (Guo & Berkhahn, 2016), which demonstrated for the first time that embeddings could systematically beat one-hot encoding for real-world tabular tasks. The advantage is largest for features with cardinality in the hundreds — things like zip codes, product IDs, or job categories — where one-hot explodes dimensionality and embeddings can capture meaningful clustering.

We demonstrate the architecture on a synthetic dataset with two continuous features and one high-cardinality categorical feature:

In [ ]:
# Synthetic dataset: y depends on continuous features + a categorical group effect
rng = np.random.default_rng(SEED)
N = 4000
C_CARDINALITY = 50                             # 50 category levels

cat_col   = rng.integers(0, C_CARDINALITY, size=N).astype(np.int64)
group_eff = np.sin(2 * np.pi * cat_col / C_CARDINALITY).astype(np.float32)  # smooth effect
x_cont    = rng.standard_normal((N, 2)).astype(np.float32)
y_synth   = x_cont[:, 0] - 0.5 * x_cont[:, 1] + 2.0 * group_eff + 0.2 * rng.standard_normal(N).astype(np.float32)

idx = np.arange(N)
rng.shuffle(idx)
n_train = int(0.8 * N)
tr, te = idx[:n_train], idx[n_train:]

sc_synth = StandardScaler()
x_cont_tr = sc_synth.fit_transform(x_cont[tr])
x_cont_te = sc_synth.transform(x_cont[te])

print(f"Synth dataset — train: {n_train}, test: {N - n_train}, categories: {C_CARDINALITY}")

**Model.** The `EmbeddingMLP` takes continuous features and categorical indices as separate inputs, embeds each categorical column, then concatenates before the MLP trunk:

In [ ]:
class EmbeddingMLP(nn.Module):
    """MLP with learned embeddings for categorical features."""

    def __init__(self, cat_cardinalities: list[int], d_cont: int,
                 hidden: int = 128, n_layers: int = 2, dropout: float = 0.1):
        super().__init__()

        # one embedding table per categorical feature                # <1>
        self.embeddings = nn.ModuleList([
            nn.Embedding(c, min(600, max(1, c // 2)))
            for c in cat_cardinalities
        ])
        d_emb = sum(min(600, max(1, c // 2)) for c in cat_cardinalities)
        d_in  = d_emb + d_cont                                       # <2>

        layers = []
        for _ in range(n_layers):
            layers += [nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden),
                       nn.ReLU(), nn.Dropout(dropout)]
            d_in = hidden
        layers.append(nn.Linear(hidden, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_cont: torch.Tensor, x_cat: torch.Tensor) -> torch.Tensor:
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]  # <3>
        x = torch.cat(embs + [x_cont], dim=1)                       # <4>
        return self.mlp(x)

1. Each categorical feature gets its own `nn.Embedding` with width $\min(600, \lfloor C/2 \rfloor)$.
2. The combined input dimension is the sum of all embedding widths plus the number of continuous features.
3. We index into each embedding table using the integer category indices for that column.
4. Embeddings are concatenated with the standardized continuous features before the MLP.

Training the embedding model on the synthetic dataset:

In [ ]:
emb_model = EmbeddingMLP(
    cat_cardinalities=[C_CARDINALITY],
    d_cont=2,
).to(DEVICE)

Xc_tr = torch.tensor(x_cont_tr, device=DEVICE)
Xi_tr = torch.tensor(cat_col[tr], device=DEVICE).unsqueeze(1)   # integer indices
yt_tr = torch.tensor(y_synth[tr], device=DEVICE).unsqueeze(1)

Xc_te = torch.tensor(x_cont_te, device=DEVICE)
Xi_te = torch.tensor(cat_col[te], device=DEVICE).unsqueeze(1)

opt2 = torch.optim.AdamW(emb_model.parameters(), lr=3e-3, weight_decay=1e-4)
mse  = nn.MSELoss()

for epoch in range(100):
    emb_model.train()
    opt2.zero_grad()
    pred = emb_model(Xc_tr, Xi_tr)
    loss = mse(pred, yt_tr)
    loss.backward()
    opt2.step()

emb_model.eval()
with torch.no_grad():
    pred_te = emb_model(Xc_te, Xi_te).squeeze().cpu().numpy()

emb_rmse = root_mean_squared_error(y_synth[te], pred_te)
print(f"EmbeddingMLP test RMSE: {emb_rmse:.4f}")

We can inspect the learned embedding space for the categorical feature. Since the ground-truth group effect is $\sin(2\pi c / C)$, categories that are close in this sinusoidal sense should cluster together in embedding space:

In [ ]:
#| code-fold: true
from sklearn.decomposition import PCA

E = emb_model.embeddings[0].weight.detach().cpu().numpy()   # (C, d_e)
pca = PCA(n_components=2)
E2 = pca.fit_transform(E)

cat_ids = np.arange(C_CARDINALITY)
true_effect = np.sin(2 * np.pi * cat_ids / C_CARDINALITY)

plt.figure(figsize=(5, 4))
sc = plt.scatter(E2[:, 0], E2[:, 1], c=true_effect, cmap="RdBu", s=50, edgecolors="k", linewidths=0.4)
plt.colorbar(sc, label="true group effect")
plt.xlabel("PC 1"); plt.ylabel("PC 2")
plt.title("Embedding space (PCA)")
plt.grid(linestyle="dotted", alpha=0.6);

**Figure.** Categories are arranged approximately in order of their true sinusoidal group effect (red = positive, blue = negative). The embedding has recovered a one-dimensional manifold that tracks the latent structure — something that would be invisible to a one-hot representation.

## TabNet

**TabNet** (Arik & Pfister, 2021) addresses the MLP's lack of native feature selection by introducing a sequential attention mechanism that selects a sparse subset of features at each processing step. The intuition is that for any given prediction, only a small number of features are truly relevant — and the model should attend to different features at different stages of its computation, much like a decision path through a tree.

**Architecture.** A TabNet network processes input $\mathbf{x} \in \mathbb{R}^d$ through $T$ sequential steps. At step $t$, an attention transformer produces a soft mask $\mathbf{m}^{(t)} \in \mathbb{R}^d$ that determines which features to attend to. The masked features $\mathbf{m}^{(t)} \odot \mathbf{x}$ are then processed by a feature transformer (a shared + step-specific dense block), and the step's output is aggregated into the final representation:

$$\mathbf{h}^{(t)} = f_t\!\left(\mathbf{m}^{(t)} \odot \mathbf{x}\right), \qquad \text{output} = \sum_{t=1}^{T} \text{ReLU}\!\left(\mathbf{h}^{(t)}\right).$$

**Sparsemax.** The attention mask is produced via **sparsemax** rather than softmax. Softmax always assigns positive probability to every feature, while sparsemax projects onto the probability simplex with $\ell_1$ constraints, producing exactly sparse outputs:

$$\text{sparsemax}(\mathbf{z}) = \underset{\mathbf{p} \in \Delta^{d-1}}{\operatorname{argmin}} \, \|\mathbf{p} - \mathbf{z}\|_2^2$$

where $\Delta^{d-1}$ is the $(d-1)$-dimensional probability simplex. The solution sets many coordinates to exactly zero, yielding a hard feature selection rather than a soft reweighting. An entropy regularization term $-\lambda \sum_t \sum_j \mathbf{m}_j^{(t)} \log \mathbf{m}_j^{(t)}$ is added to prevent degenerate masks that always attend to the same feature.

**Interpretability.** The cumulative attention weights $A_j = \sum_t \mathbf{m}_j^{(t)}$ across all steps give a feature importance score that is natively produced by the forward pass — no post-hoc SHAP computation required. This makes TabNet attractive in regulated settings where models must explain individual predictions.

**Self-supervised pretraining.** TabNet also supports a self-supervised pretraining phase: randomly mask a subset of input features and train the network to reconstruct them from the unmasked features. This is the tabular equivalent of masked autoencoders, and it can improve performance significantly when labeled data is scarce but unlabeled tabular rows are abundant.

**When TabNet helps.** In practice, TabNet outperforms MLPs most reliably when (1) the dataset has many irrelevant features and sparsity is a meaningful prior, (2) interpretability via attention weights is a requirement, or (3) self-supervised pretraining is available. On standard benchmarks without pretraining, it rarely beats well-tuned XGBoost, and its training is sensitive to hyperparameters (number of steps $T$, sparsity coefficient $\lambda$, feature transformer width).

The core of TabNet is the sparsemax attention and sequential masking. A minimal sketch of the step logic:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def sparsemax(z: torch.Tensor) -> torch.Tensor:
    """Sparsemax activation (1D along last dimension)."""
    z_sorted, _ = torch.sort(z, dim=-1, descending=True)
    d = z.shape[-1]
    cumsum = torch.cumsum(z_sorted, dim=-1)
    k = torch.arange(1, d + 1, device=z.device, dtype=z.dtype)
    support = z_sorted - (cumsum - 1) / k > 0                       # <1>
    k_max   = support.sum(dim=-1, keepdim=True).float()
    tau     = (cumsum.gather(-1, (k_max.long() - 1)) - 1) / k_max  # <2>
    return torch.clamp(z - tau, min=0.0)


class TabNetStep(nn.Module):
    """One sequential attention step of TabNet."""

    def __init__(self, d_in: int, d_out: int):
        super().__init__()
        self.attention = nn.Linear(d_out, d_in, bias=False)         # <3>
        self.bn_att    = nn.BatchNorm1d(d_in)
        self.feat_fc   = nn.Linear(d_in, d_out * 2)                 # <4>
        self.bn_feat   = nn.BatchNorm1d(d_out * 2)

    def forward(self, x: torch.Tensor, h_prev: torch.Tensor, prior_scales: torch.Tensor):
        # Compute sparse attention mask
        att_logits = self.bn_att(self.attention(h_prev))
        mask = sparsemax(att_logits * prior_scales)                  # <5>

        # Masked feature processing
        h = self.bn_feat(self.feat_fc(mask * x))
        h1, h2 = h.chunk(2, dim=-1)                                 # <6>
        h_out  = torch.sqrt(torch.tensor(0.5)) * (h1 * torch.sigmoid(h2) + h1)
        return h_out, mask

1. The support set is the set of indices where $z_{(k)} > (\text{cumsum}_k - 1)/k$, identifying which coordinates of $\mathbf{z}$ receive nonzero probability under sparsemax.
2. The threshold $\tau$ is the Lagrange multiplier that enforces the simplex constraint; subtracting it and clamping to zero gives the sparse output.
3. The attention network projects the previous step's hidden state back to feature space to produce the mask logits.
4. The feature transformer has width $2 \times d_{\text{out}}$ to support the gated linear unit activation used in the original paper.
5. `prior_scales` tracks which features have been used in previous steps and penalizes reuse — this encourages the network to attend to different features at each step.
6. The gated linear unit splits the output into two halves and applies $\text{GLU}(h_1, h_2) = h_1 \odot \sigma(h_2) + h_1$, mixing linear and sigmoidal paths.

## FT-Transformer

**FT-Transformer** (Gorishniy et al., 2021) applies a standard Transformer encoder to tabular data by treating each feature as a token. The key insight is that self-attention over feature tokens can model arbitrary pairwise feature interactions in a way that is symmetric and does not depend on the order features appear in the table — properties that MLPs (which depend on the ordering of the input vector) lack.

**Tokenization.** Each continuous feature $x_j \in \mathbb{R}$ is converted to a token $\mathbf{t}_j = x_j \cdot \mathbf{w}_j + \mathbf{b}_j \in \mathbb{R}^{d}$ via a learned linear projection, where $\mathbf{w}_j, \mathbf{b}_j \in \mathbb{R}^{d}$ are feature-specific parameters. Each categorical feature is tokenized via a standard `nn.Embedding` as before. A learnable `[CLS]` token $\mathbf{c} \in \mathbb{R}^d$ is prepended to the sequence. The full input to the Transformer is therefore the sequence of $d_{\text{feat}} + 1$ tokens:

$$\mathbf{T} = \left[\mathbf{c},\; \mathbf{t}_1,\; \ldots,\; \mathbf{t}_{d_{\text{feat}}}\right] \in \mathbb{R}^{(d_{\text{feat}}+1) \times d}.$$

**Encoder and readout.** The sequence $\mathbf{T}$ is processed by $L$ standard Transformer encoder layers (multi-head self-attention + feedforward). The final representation of the `[CLS]` token is extracted and passed to a linear head to produce the prediction. Pre-norm (LayerNorm before attention and feedforward) is used throughout, following the convention in language modeling.

**Empirical results.** Gorishniy et al. benchmarked FT-Transformer against a large suite of tree methods and MLPs on public tabular datasets. FT-Transformer matches or exceeds XGBoost on several datasets, particularly those with smooth target functions and many dense features. However, Grinsztajn et al. (2022) provided a more careful analysis: on datasets with irregular (piecewise-constant) targets or many uninformative features, trees consistently win. The Transformer's self-attention is powerful but isotropic — it treats all feature pairs equally before learning, whereas trees are born with a built-in ability to ignore irrelevant features.

**Trade-offs.** FT-Transformer has $O(d_{\text{feat}}^2)$ attention complexity per layer, which becomes expensive for datasets with hundreds of features. Training is substantially slower than XGBoost: a competitive FT-Transformer requires careful initialization, a warm-up learning rate schedule, and typically more than 100 epochs. The upside is that the architecture composes naturally with pretraining and transfer learning — embeddings from one tabular dataset can initialize another — which is not feasible with tree methods.

A minimal but complete FT-Transformer implementation in PyTorch:

In [ ]:
class FeatureTokenizer(nn.Module):
    """Tokenize continuous and categorical features into a shared d-dimensional space."""

    def __init__(self, d_cont: int, cat_cardinalities: list[int], d: int):
        super().__init__()
        # Continuous: one weight vector and bias vector per feature   # <1>
        self.weight_cont = nn.Parameter(torch.empty(d_cont, d))
        self.bias_cont   = nn.Parameter(torch.empty(d_cont, d))
        nn.init.kaiming_uniform_(self.weight_cont, a=np.sqrt(5))
        nn.init.zeros_(self.bias_cont)

        # Categorical: one embedding table per column                 # <2>
        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(c, d) for c in cat_cardinalities
        ])

    def forward(self, x_cont: torch.Tensor, x_cat: torch.Tensor | None = None):
        # (B, d_cont) → (B, d_cont, d)                               # <3>
        tokens = x_cont.unsqueeze(-1) * self.weight_cont + self.bias_cont

        if x_cat is not None:
            cat_tokens = torch.stack(
                [emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embeddings)],
                dim=1
            )                                                         # <4>
            tokens = torch.cat([tokens, cat_tokens], dim=1)          # (B, d_feat, d)

        return tokens


class FTTransformer(nn.Module):
    """Feature Tokenizer + Transformer (Gorishniy et al., 2021)."""

    def __init__(self, d_cont: int, cat_cardinalities: list[int],
                 d: int = 64, n_heads: int = 4, n_layers: int = 3,
                 ffn_factor: int = 4, dropout: float = 0.1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(d_cont, cat_cardinalities, d)
        n_tokens = d_cont + len(cat_cardinalities) + 1               # features + [CLS]

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d))          # <5>

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d,
            nhead=n_heads,
            dim_feedforward=d * ffn_factor,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,   # pre-norm                             # <6>
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d),
            nn.Linear(d, 1),
        )

    def forward(self, x_cont: torch.Tensor, x_cat: torch.Tensor | None = None):
        tokens = self.tokenizer(x_cont, x_cat)                       # (B, d_feat, d)
        cls = self.cls_token.expand(tokens.shape[0], -1, -1)         # (B, 1, d)
        seq = torch.cat([cls, tokens], dim=1)                        # (B, d_feat+1, d)
        out = self.encoder(seq)                                      # (B, d_feat+1, d)
        return self.head(out[:, 0])                                   # <7>

1. Each continuous feature gets its own projection: $\mathbf{t}_j = x_j \cdot \mathbf{w}_j + \mathbf{b}_j \in \mathbb{R}^d.$ The weights are stored as `(d_cont, d)` matrices for batch efficiency.
2. Categorical features are tokenized via standard `nn.Embedding` tables with dimension $d$, putting them in the same space as the continuous tokens.
3. Broadcasting: `x_cont` has shape `(B, d_cont)`, so `x_cont.unsqueeze(-1)` gives `(B, d_cont, 1)`, and multiplying by `weight_cont` of shape `(d_cont, d)` gives `(B, d_cont, d)` tokens.
4. Category tokens are stacked along the sequence dimension to produce `(B, n_cat, d)` before concatenation.
5. The `[CLS]` token is a learnable parameter initialized to zero; it is expanded to the batch size during the forward pass.
6. Pre-norm (`norm_first=True`) applies LayerNorm before attention and feedforward, which stabilizes training in deeper Transformers.
7. Only the `[CLS]` token's final hidden state is used for prediction — the per-feature tokens are discarded after encoding.

Verifying the model on a forward pass with the California housing feature dimensions:

In [ ]:
ft = FTTransformer(d_cont=8, cat_cardinalities=[], d=64, n_heads=4, n_layers=3).to(DEVICE)

dummy = torch.randn(32, 8, device=DEVICE)   # batch of 32, 8 continuous features
out   = ft(dummy)
print(f"output shape: {out.shape}")         # expect (32, 1)

n_params = sum(p.numel() for p in ft.parameters())
print(f"FT-Transformer parameters: {n_params:,}")

**Training FT-Transformer** on California housing to compare with the MLP baseline:

In [ ]:
ft_model = FTTransformer(d_cont=8, cat_cardinalities=[], d=64, n_heads=4, n_layers=3).to(DEVICE)

opt_ft = torch.optim.AdamW(ft_model.parameters(), lr=1e-4, weight_decay=1e-5)
sched_ft = torch.optim.lr_scheduler.CosineAnnealingLR(opt_ft, T_max=100, eta_min=1e-6)

best_ft_rmse = float("inf")
best_ft_state = None
patience_ft = 0

n = Xtr.shape[0]
for epoch in range(100):
    ft_model.train()
    perm = torch.randperm(n, device=DEVICE)
    for i in range(0, n, BATCH):
        idx = perm[i:i + BATCH]
        xb, yb = Xtr[idx], ytr[idx]
        opt_ft.zero_grad()
        loss = criterion(ft_model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)         # <1>
        opt_ft.step()
    sched_ft.step()

    ft_model.eval()
    with torch.no_grad():
        vp = ft_model(Xvl).squeeze().cpu().numpy()
    vr = root_mean_squared_error(y_val, vp)

    if vr < best_ft_rmse:
        best_ft_rmse = vr
        best_ft_state = {k: v.clone() for k, v in ft_model.state_dict().items()}
        patience_ft = 0
    else:
        patience_ft += 1
        if patience_ft >= 15:
            print(f"Early stop at epoch {epoch + 1}")
            break

ft_model.load_state_dict(best_ft_state)
ft_model.eval()
with torch.no_grad():
    test_pred_ft = ft_model(Xte).squeeze().cpu().numpy()

ft_rmse = root_mean_squared_error(y_test, test_pred_ft)
print(f"FT-Transformer test RMSE: {ft_rmse:.4f}")

1. Gradient clipping to norm 1.0 is standard practice for Transformer training; without it, occasional large gradients can destabilize the attention weights early in training.

## When Do Neural Networks Beat Trees?

Three large-scale benchmarking papers have clarified the empirical picture. **Gorishniy et al. (2021)** benchmarked FT-Transformer and a simple ResNet-style MLP against XGBoost on 11 public datasets, finding that FT-Transformer was competitive on several but not dominant overall. **Grinsztajn et al. (2022)** conducted a controlled study on 45 datasets and isolated the key predictor of tree dominance: datasets where the target function has **irregular (non-smooth) boundaries** aligned with individual feature axes. Their analysis showed that trees are especially difficult to dethrone on datasets with many uninformative features, because their built-in feature selection prevents irrelevant dimensions from contaminating splits. **McElfresh et al. (2023)** ran an exhaustive sweep across 19 methods and 176 datasets, concluding that no single method dominates across all dataset types, and that proper hyperparameter tuning narrows the gap significantly — but XGBoost remains the single most reliable first choice.

Pulling these studies together, the pattern is consistent. Trees dominate when (1) the dataset has fewer than $\sim$100K samples, (2) the target function has sharp irregular boundaries (piecewise-constant behavior), (3) many features are uninformative or only weakly correlated with the target, or (4) categorical features have low to medium cardinality where one-hot encoding is feasible. Neural methods become competitive or dominant when (5) the dataset exceeds $\sim$500K samples and batch optimization produces reliable gradient estimates, (6) the target function is smooth (think: regression on physical quantities with underlying continuous generative processes), (7) categorical features have very high cardinality (zip codes, product IDs, user IDs) where embeddings generalize better, or (8) pretraining on unlabeled tabular data or transfer from a related task is possible.

[The practitioner's decision is almost never a theoretical one — it is a question of where you are in the data and compute budget.]{.mark}

**If your dataset has fewer than 50K samples:** Use XGBoost or LightGBM with cross-validated hyperparameter search. Neural methods will overfit unless you invest heavily in regularization and architecture search.

**If your dataset has more than 500K samples:** Both trees and neural methods are viable. Try XGBoost first (it is faster to iterate), then benchmark an MLP with embeddings. Consider FT-Transformer only if you have evidence the dataset has smooth interactions.

**If you have many high-cardinality categorical features:** Embeddings are likely to outperform one-hot baselines. An EmbeddingMLP or FT-Transformer (which tokenizes each feature) is a natural choice.

**If interpretability is a hard requirement:** Tree SHAP (via `shap.TreeExplainer`) is computationally exact and fast. TabNet's attention weights provide a native but noisier alternative.

**If you have access to unlabeled tabular data at scale:** TabNet's masked pretraining or a masked autoencoder on feature tokens can produce significant improvements. This is the regime where neural methods have a structural advantage that trees cannot exploit.

**If you are building a system that must retrain frequently on new data:** XGBoost retrains orders of magnitude faster than any neural method and requires no GPU. This operational consideration alone often determines the answer in production.

:::{.callout-note}
The meta-learning literature (e.g., TabPFN by Hollmann et al., 2023) offers a different escape hatch: train a Transformer *across* many tabular datasets as a prior, then use it in-context at inference time without any dataset-specific training. TabPFN achieves strong results on small datasets (up to a few thousand rows) in under a second of inference time — a regime where XGBoost requires careful cross-validation. This is an active research frontier, and the boundaries between these methods continue to shift.

:::

The benchmark figure below compares model classes across three sklearn datasets, normalized so the best model per dataset equals 1.0:

In [ ]:
#| label: fig-tabular-benchmark
#| fig-cap: "Relative performance of model classes across three tabular datasets (best model per dataset = 1.0; lower RMSE or higher accuracy is better). Error bars show one standard deviation across 5-fold cross-validation."
#| code-fold: true

from sklearn.datasets import fetch_california_housing, fetch_covtype
from sklearn.datasets import fetch_openml
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor, XGBClassifier
import numpy as np

# ── Dataset definitions ──────────────────────────────────────────────
datasets_def = {
    "CalifHousing": ("regression",),
    "Adult": ("classification",),
    "CovType": ("classification",),
}

def load_datasets():
    out = {}
    d = fetch_california_housing()
    out["CalifHousing"] = (d.data, d.target, "regression")

    adult = fetch_openml("adult", version=2, as_frame=False)
    X_a = adult.data.astype(np.float32)
    y_a = (adult.target == ">50K").astype(int)
    out["Adult"] = (X_a, y_a, "classification")

    cov = fetch_covtype()
    out["CovType"] = (cov.data[:20000], cov.target[:20000] - 1, "classification")
    return out

datasets = load_datasets()

# ── MLP training function (no cross-val, quick single-split eval) ────
def eval_mlp_cv(X, y, task, n_splits=5, seed=42):
    """Simple CV for MLP with StandardScaler baked in."""
    rng_cv = np.random.default_rng(seed)
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=seed) if task == "regression" \
        else StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    scores = []
    for tr_idx, te_idx in cv.split(X, y):
        Xtr_, Xte_ = X[tr_idx], X[te_idx]
        ytr_, yte_ = y[tr_idx], y[te_idx]
        sc_ = StandardScaler()
        Xtr_ = sc_.fit_transform(Xtr_).astype(np.float32)
        Xte_ = sc_.transform(Xte_).astype(np.float32)

        d_in = Xtr_.shape[1]
        n_cls = len(np.unique(ytr_)) if task == "classification" else 1
        net = make_mlp(d_in=d_in, hidden=256, n_layers=3, dropout=0.1).to(DEVICE)
        if task == "classification":
            net[-1] = nn.Linear(256, n_cls).to(DEVICE)

        opt_ = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
        sched_ = torch.optim.lr_scheduler.CosineAnnealingLR(opt_, T_max=50)
        crit_ = nn.MSELoss() if task == "regression" else nn.CrossEntropyLoss()

        Xt = torch.tensor(Xtr_, device=DEVICE)
        yt = torch.tensor(ytr_, device=DEVICE)
        yt = yt.unsqueeze(1).float() if task == "regression" else yt.long()

        n_ = Xt.shape[0]
        for ep in range(50):
            net.train()
            perm_ = torch.randperm(n_, device=DEVICE)
            for i_ in range(0, n_, 512):
                idx_ = perm_[i_:i_ + 512]
                opt_.zero_grad()
                crit_(net(Xt[idx_]), yt[idx_]).backward()
                opt_.step()
            sched_.step()

        net.eval()
        Xte_t = torch.tensor(Xte_, device=DEVICE)
        with torch.no_grad():
            pred_ = net(Xte_t).cpu().numpy()

        if task == "regression":
            scores.append(root_mean_squared_error(yte_, pred_.squeeze()))
        else:
            pred_cls = pred_.argmax(1)
            scores.append((pred_cls == yte_).mean())
    return np.array(scores)

# ── Cross-validation for sklearn models ─────────────────────────────
results_raw = {}   # dataset -> model -> (mean, std)

for ds_name, (X, y, task) in datasets.items():
    results_raw[ds_name] = {}
    cv = KFold(5, shuffle=True, random_state=42) if task == "regression" \
        else StratifiedKFold(5, shuffle=True, random_state=42)
    scoring = "neg_root_mean_squared_error" if task == "regression" else "accuracy"

    # Linear baseline
    lin = Pipeline([("sc", StandardScaler()),
                    ("m", Ridge() if task == "regression" else LogisticRegression(max_iter=1000))])
    s = cross_val_score(lin, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    results_raw[ds_name]["Linear"] = s if task == "classification" else -s

    # Random Forest
    rf_ = RandomForestRegressor(200, n_jobs=-1, random_state=42) if task == "regression" \
        else RandomForestClassifier(200, n_jobs=-1, random_state=42)
    s = cross_val_score(rf_, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    results_raw[ds_name]["RandomForest"] = s if task == "classification" else -s

    # XGBoost
    xgb_ = XGBRegressor(n_estimators=300, learning_rate=0.05, verbosity=0, random_state=42) \
        if task == "regression" else \
        XGBClassifier(n_estimators=300, learning_rate=0.05, verbosity=0,
                      use_label_encoder=False, eval_metric="logloss", random_state=42)
    s = cross_val_score(xgb_, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    results_raw[ds_name]["XGBoost"] = s if task == "classification" else -s

    # MLP
    mlp_scores = eval_mlp_cv(X, y, task)
    results_raw[ds_name]["MLP"] = mlp_scores

# ── Normalize per dataset (best = 1.0) ──────────────────────────────
model_names  = ["Linear", "RandomForest", "XGBoost", "MLP"]
colors       = ["C0", "C2", "C1", "C3"]
ds_names     = list(results_raw.keys())
n_ds         = len(ds_names)
n_models     = len(model_names)

means_norm = np.zeros((n_ds, n_models))
stds_norm  = np.zeros((n_ds, n_models))

for i, ds in enumerate(ds_names):
    task_ = datasets[ds][2]
    raw_means = np.array([results_raw[ds][m].mean() for m in model_names])
    raw_stds  = np.array([results_raw[ds][m].std()  for m in model_names])
    best      = raw_means.max() if task_ == "classification" else raw_means.min()
    # For accuracy: higher is better; normalize to [0,1] where 1 = best
    # For RMSE: lower is better; normalize so best = 1.0 (invert)
    if task_ == "classification":
        means_norm[i] = raw_means / best
        stds_norm[i]  = raw_stds  / best
    else:
        means_norm[i] = best / raw_means
        stds_norm[i]  = raw_stds / best

# ── Grouped bar chart ────────────────────────────────────────────────
x       = np.arange(n_ds)
width   = 0.18
offsets = np.linspace(-(n_models - 1) / 2, (n_models - 1) / 2, n_models) * width

fig, ax = plt.subplots(figsize=(8, 4))
for j, (model, color, offset) in enumerate(zip(model_names, colors, offsets)):
    ax.bar(
        x + offset, means_norm[:, j], width,
        yerr=stds_norm[:, j], capsize=3,
        color=color, alpha=0.85, label=model, error_kw={"linewidth": 1.0}
    )

ax.axhline(1.0, color="black", linestyle="--", linewidth=0.8, alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels([f"{n}\n({datasets[n][2]})" for n in ds_names])
ax.set_ylabel("Relative performance\n(best model = 1.0)")
ax.set_ylim(0, 1.12)
ax.legend(loc="lower right", framealpha=0.9)
ax.grid(axis="y", linestyle="dotted", alpha=0.6)
ax.set_title("Tabular benchmark: model class comparison")
plt.tight_layout();

**Figure.** XGBoost and Random Forest consistently reach near-peak performance across all three datasets. The MLP — properly configured with BatchNorm, AdamW, and cosine scheduling — is competitive but trails on California housing (a medium-sized regression with moderately irregular target) and CovType (a classification task with axis-aligned decision boundaries that trees handle naturally). The gap narrows when the dataset grows and the target smooths out, which is the regime where FT-Transformer begins to close the distance.

---

■